# Git and GitHub

**Time:** 45–60 minutes  
**Prerequisites:** Git installed, GitHub account connected to the ALL AI Network workspace.

## Learning objectives

- Inspect repository state without changing anything
- Create branches, commits, and pull requests safely
- Understand what must **not** be committed (weights, `.venv`, `runs/`, secrets)
- Resolve a simple merge conflict in practice

## Checkpoint

You can explain the difference between `git status`, `git diff`, and `git log`, and you know where this repo’s `.gitignore` rules live.

## Find the repository root

In [ ]:
from pathlib import Path

def repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").is_dir():
            return parent
    raise FileNotFoundError("Not inside a git repository")

ROOT = repo_root()
SIM = ROOT / "fanuc_er4ia_sim"
print("Repository root:", ROOT)
print("Simulation package:", SIM)

## Read-only inspection (safe anytime)

Run these in a terminal from the repository root. They do not modify history.

```bash
git status
git log --oneline -5
git diff
git remote -v
git branch -a
```

**Exercise:** Run `git status`. If you have uncommitted tutorial work, that is expected while learning.

In [ ]:
import subprocess

def git(*args: str) -> str:
    return subprocess.check_output(
        ["git", *args], cwd=ROOT, text=True, stderr=subprocess.STDOUT
    )

print(git("status", "-sb"))
print("--- recent commits ---")
print(git("log", "--oneline", "-5"))

## What not to commit

The simulation starter ignores local artifacts in [`fanuc_er4ia_sim/.gitignore`](../../fanuc_er4ia_sim/.gitignore):

- `.venv/`, `__pycache__/`
- `runs/`, captured frames, `*.mp4`
- Large zips and gripper shot folders

**Never commit:** API keys, `.env` files with secrets, Cosmos model weights, Isaac caches.

**TODO (discussion):** Open `.gitignore` and list three paths that would be harmful to push.

In [ ]:
ignore = (SIM / ".gitignore").read_text(encoding="utf-8")
print(ignore)

## Branching workflow

Typical team flow:

1. `git pull origin main`
2. `git switch -c feature/my-change`
3. Edit, test, `git add` specific files, `git commit`
4. `git push -u origin feature/my-change`
5. Open a **pull request** on GitHub for review

**Exercise (mutating — use a throwaway branch):**

```bash
git switch -c tutorial/git-practice
# make a one-line change in a scratch file, then:
git add tutorials/notebooks/.git_practice_scratch.txt
git commit -m "docs: git practice scratch"
git push -u origin tutorial/git-practice
```

Create the PR in the GitHub UI. Close or merge only after your mentor agrees.

**Cleanup:** delete the branch locally and on GitHub when done.

## Pull requests and review

Pull requests let mentors see diffs before `main` changes. Good PR descriptions include:

- What problem you solved
- How you tested
- Screenshots or log snippets for sim runs

**Can't push?** Use the GitHub check button in the [team workspace](https://dashboard.all-ai-network.org/j/L36HEH) (see root README).

## Merge conflicts (mini exercise)

Conflicts happen when two branches edit the same lines.

1. On branch A, edit a line in a text file.
2. On branch B (from `main`), edit the same line differently.
3. Merge B into A — Git marks conflict markers `<<<<<<<`.
4. Edit to the correct combined text, `git add`, `git commit`.

Search for conflict markers: `git diff --check`.

## Troubleshooting

| Issue | Fix |
|-------|-----|
| `permission denied (publickey)` | SSH keys or HTTPS credential helper |
| `rejected (fetch first)` | `git pull --rebase origin main` then push |
| Accidentally staged a huge file | `git restore --staged <file>` before commit |

## Cleanup

- Remove practice branches and scratch commits you do not need.
- Never force-push to `main` without explicit team approval.